# BEB-La-DII: Phase 1 Training (Reasoning)
Этот блокнот оптимизирован для второй стадии (Reasoning). Он включает в себя:
1. Автоматическую настройку зеркальной структуры путей.
2. Инсталляцию зависимостей.
3. Загрузку весов Awakening из кастомного датасета.

In [ ]:
# 0. ПОЛУЧЕНИЕ И ОБНОВЛЕНИЕ ИСХОДНОГО КОДА
import os, sys, shutil
from pathlib import Path

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/Laeryid/BEBLaDII"
REPO_NAME = "BEBLaDII"

os.chdir("/kaggle/working/")

if not os.path.exists(REPO_NAME):
    print(f"Клонирование репозитория {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print(f"Репозиторий {REPO_NAME} уже присутствует. Проверка обновлений...")
    !cd {REPO_NAME} && git pull

if os.path.exists(REPO_NAME) and REPO_NAME not in os.getcwd():
    os.chdir(REPO_NAME)
    print(f"Рабочая директория: {os.getcwd()}")

In [ ]:
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ И ПУТЕЙ
import subprocess
def install_packages():
    packages = ["transformers==4.57.2", "indexed-parquet-dataset", "optimum-intel[openvino]", "wandb", "accelerate", "bitsandbytes"]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_packages()

root_str = os.getcwd()
if root_str not in sys.path: sys.path.insert(0, root_str)
print(f"Корень проекта: {root_str}")

In [ ]:
import os, torch, json, wandb
from torch.optim import AdamW
from tqdm.auto import tqdm

try:
    from src.beb_la_dii.model.assembler import ModelAssembler
    from src.beb_la_dii.utils.loss import DistillationLoss
    from src.beb_la_dii.utils.data import get_dataloader
    from src.beb_la_dii.utils.experiment_tracker import ExperimentTracker
    from src.beb_la_dii.utils.tokenizer import get_tokenizer
    print("Модули проекта успешно импортированы.")
except ImportError as e: print(f"Ошибка импорта: {e}")

In [ ]:
# --- PATH CONFIGURATION ---
VERSION = "v1.0"
KAGGLE_RESOURCES_DATASET = "/kaggle/input/datasets/bogdanbuliakov/bebladii-resources"
KAGGLE_TEACHER_MODEL = "/kaggle/input/models/deepseek-ai/deepseek-r1/transformers/deepseek-r1-distill-qwen-7b/2"

TEACHER_NAME = KAGGLE_TEACHER_MODEL if os.path.exists(KAGGLE_TEACHER_MODEL) else "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
RESOURCES_PATH = KAGGLE_RESOURCES_DATASET if os.path.exists(KAGGLE_RESOURCES_DATASET) else "./storage"

# Hyperparameters
BASE_MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LENGTH = 4096
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
STAGE = 'reasoning'
VAL_EVERY_STEPS = 200
VAL_MAX_SAMPLES = 100
LEARNING_RATE = 2e-5
EPOCHS = 1
WARMUP_STEPS = 32

CUSTOM_STUDENT_WEIGHTS_PATH = "/kaggle/input/datasets/bogdanbuliakov/bebladii-phase1-awakaned-weights/AWAKENED_WEIGHTS_FINAL.pt"

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
def setup_mirrored_kaggle(resources_path):
    """Настройка зеркальной структуры симлинков"""
    if not os.path.exists("/kaggle/input"): return
    print("--- Зеркальная настройка ресурсов Kaggle ---")
    
    if not os.path.exists(resources_path):
        print(f"WARN: Датасет ресурсов не найден по пути {resources_path}")
        return
        
    mappings = [
        ("components", "storage/components"), 
        ("prebuilt", "storage/prebuilt"), 
        ("data", "data")
    ]
    
    for src_name, dst_path in mappings:
        src = os.path.join(resources_path, src_name)
        dst = os.path.abspath(dst_path)
        if os.path.exists(src):
            if os.path.exists(dst):
                if os.path.islink(dst): os.unlink(dst)
                else: shutil.rmtree(dst)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            os.symlink(src, dst)
            print(f"УСПЕХ: {dst_path} -> {src_name}")

setup_mirrored_kaggle(RESOURCES_PATH)

In [ ]:
# СБОРКА МОДЕЛИ И НАСТРОЙКА ГРАДИЕНТОВ
from experiments.train_phase1_kaggle import build_weights_map

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Инициализация на {device}...")

# Определение путей
if os.path.exists(KAGGLE_RESOURCES_DATASET):
    student_base_id = os.path.join(KAGGLE_RESOURCES_DATASET, "prebuilt/latentBERT", VERSION)
    components_root = os.path.join(KAGGLE_RESOURCES_DATASET, "components")
else:
    student_base_id = os.path.join("storage/prebuilt/latentBERT", VERSION)
    if not os.path.exists(student_base_id): student_base_id = BASE_MODEL_NAME
    components_root = "storage/components"

weights_map = build_weights_map(components_root=components_root)
assembler = ModelAssembler()
distiller = assembler.assemble_phase1_distiller(
    teacher_id=TEACHER_NAME, 
    student_base_id=student_base_id,
    version=VERSION,
    weights_map=weights_map,
    device_map={"": 0},
    student_device="cuda:1"
)

# Оптимизации памяти для длинных последовательностей
if hasattr(distiller.student.model, 'gradient_checkpointing_enable'):
    print('Enabling Gradient Checkpointing for Student...')
    distiller.student.model.gradient_checkpointing_enable()
    distiller.student.model.config.use_cache = False

# --- ЗАГРУЗКА КАСТОМНЫХ ВЕСОВ ---
if CUSTOM_STUDENT_WEIGHTS_PATH and os.path.exists(CUSTOM_STUDENT_WEIGHTS_PATH):
    print(f"Загрузка кастомных весов студента из {CUSTOM_STUDENT_WEIGHTS_PATH}...")
    ckpt = torch.load(CUSTOM_STUDENT_WEIGHTS_PATH, map_location='cpu')
    if isinstance(ckpt, dict) and "latentBERT_state_dict" in ckpt:
        distiller.student.load_state_dict(ckpt["latentBERT_state_dict"])
        if "input_projector" in ckpt:
            distiller.input_projector.load_state_dict(ckpt["input_projector"])
        if "feature_projectors" in ckpt:
            distiller.feature_projectors.load_state_dict(ckpt["feature_projectors"])
        print("Успешно: Загружен полный чекпоинт")
    else:
        distiller.student.load_state_dict(ckpt)
        print("Успешно: Загружен state_dict студента")

# 1. Замораживаем всё
for p in distiller.parameters(): p.requires_grad = False

# 2. Размораживаем студента и проекторы
for p in distiller.student.parameters(): p.requires_grad = True
for p in distiller.input_projector.parameters(): p.requires_grad = True
for proj in distiller.feature_projectors.values():
    for p in proj.parameters(): p.requires_grad = True

print(f"Обучаемых параметров: {sum(p.numel() for p in distiller.parameters() if p.requires_grad):,}")

In [ ]:
# ПОДГОТОВКА ОБУЧЕНИЯ
from kaggle_secrets import UserSecretsClient

tokenizer = get_tokenizer()
train_loader = get_dataloader(stage=STAGE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH, split='train')
val_loader = get_dataloader(stage=STAGE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH, split='val')
optimizer = AdamW(filter(lambda p: p.requires_grad, distiller.parameters()), lr=LEARNING_RATE)
criterion = DistillationLoss()
# Добавляем скейлер для Mixed Precision
scaler = torch.amp.GradScaler('cuda')
tracker = ExperimentTracker(project_root=".", stage=STAGE)

try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    if wandb_key: os.environ["WANDB_API_KEY"] = wandb_key
except Exception: pass

if os.environ.get("WANDB_API_KEY"):
    wandb.init(project="BEBLaDII", name=f"phase1-{STAGE}")

In [ ]:
# ЦИКЛ ОБУЧЕНИЯ
distiller.train()
print(f"--- Запуск обучения Фазы 1 ({STAGE}) ---")

for epoch in range(EPOCHS):
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}")
    accum_loss = 0.0
    accum_mse = 0.0
    accum_cosine = 0.0
    accum_metrics = {} # Для послойных метрик
    
    # Сброс градиентов перед началом эпохи
    optimizer.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        try:
            # 1. Используем distiller.student_device вместо глобального device.
            # Это гарантирует, что данные попадут на ту же GPU, что и модель студента.
            input_ids = batch['input_ids'].to(distiller.student_device).to(torch.long)
            mask = batch['attention_mask'].to(distiller.student_device)
            
            with torch.amp.autocast('cuda'):
                student_states, teacher_targets = distiller(input_ids, mask)
                loss_mask = mask.to(distiller.student_device)
                loss, loss_metrics = criterion(student_states, teacher_targets, attention_mask=loss_mask)
                loss = loss / GRAD_ACCUM_STEPS
            
            # 2. Мягкая защита от NaN. Если лосс "вылетел", мы не вызываем .backward(),
            # чтобы не отравить веса бесконечными градиентами.
            if torch.isnan(loss):
                print(f"\n[WARN] NaN loss detected at step {step}. Skipping batch.")
                optimizer.zero_grad() # Сбрасываем накопленное в этом батче
                continue
            
            # 3. Scale Backward. Это критически важно для FP16. 
            # Скейлер увеличивает лосс перед бэквардом, чтобы малые градиенты не превратились в нули.
            scaler.scale(loss).backward()
            
            accum_loss += loss.item()
            accum_mse += loss_metrics['mse'] / GRAD_ACCUM_STEPS
            accum_cosine += loss_metrics['cosine'] / GRAD_ACCUM_STEPS
            
            for k, v in loss_metrics.items():
                if k.startswith("l") and ("_mse" in k or "_cos" in k):
                    accum_metrics[k] = accum_metrics.get(k, 0.0) + v / GRAD_ACCUM_STEPS
            
            # 4. Шаг оптимизатора (выполняется раз в GRAD_ACCUM_STEPS)
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                # Внутренний Warmup (линейный рост LR в начале обучения)
                macro_step = (step + 1) // GRAD_ACCUM_STEPS
                if macro_step <= WARMUP_STEPS:
                    lr_scale = macro_step / WARMUP_STEPS
                    for pg in optimizer.param_groups: 
                        pg['lr'] = LEARNING_RATE * lr_scale
                
                # 5. Unscale перед клиппингом. Скейлер возвращает градиенты к нормальному масштабу,
                # чтобы torch.nn.utils.clip_grad_norm_ корректно их обрезал.
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(distiller.parameters(), 1.0)
                
                # 6. Безопасный шаг через скейлер. Если в градиентах есть NaN/Inf, 
                # скейлер просто пропустит этот шаг (update не произойдет).
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
                avg_loss = accum_loss
                avg_mse = accum_mse
                avg_cos = accum_cosine
                current_lr = optimizer.param_groups[0]['lr']
                
                progress_bar.set_postfix({"loss": f"{avg_loss:.4f}", "mse": f"{avg_mse:.4f}", "gn": f"{grad_norm:.2f}"})
                
                if wandb.run: 
                    log_dict = {
                        "loss": avg_loss, 
                        "mse": avg_mse,
                        "cosine": avg_cos,
                        "grad_norm": grad_norm.item() if torch.is_tensor(grad_norm) else grad_norm,
                        "step": step,
                        "lr": current_lr
                    }
                    for k, v in accum_metrics.items(): log_dict[f"train/{k}"] = v
                    wandb.log(log_dict)
                
                # Обнуляем аккумуляторы метрик
                accum_loss = 0.0; accum_mse = 0.0; accum_cosine = 0.0; accum_metrics = {}
            
            # --- ВАЛИДАЦИЯ ---
            if (step + 1) % VAL_EVERY_STEPS == 0:
                print(f"\n--- Валидация (Шаг {step+1}) ---")
                distiller.eval()
                val_loss_sum = 0.0
                val_mse_sum = 0.0
                val_cos_sum = 0.0
                val_steps = 0
                max_val_steps = VAL_MAX_SAMPLES // BATCH_SIZE
                
                with torch.no_grad():
                    for v_step, v_batch in enumerate(val_loader):
                        if v_step >= max_val_steps: break
                        v_input_ids = v_batch['input_ids'].to(distiller.student_device).to(torch.long)
                        v_mask = v_batch['attention_mask'].to(distiller.student_device)
                        
                        with torch.amp.autocast('cuda'):
                            v_st, v_tgt = distiller(v_input_ids, v_mask)
                            v_loss_msk = v_mask.to(distiller.student_device)
                            v_loss, v_metrics = criterion(v_st, v_tgt, attention_mask=v_loss_msk)
                        
                        val_loss_sum += v_loss.item()
                        val_mse_sum += v_metrics["mse"]
                        val_cos_sum += v_metrics["cosine"]
                        val_steps += 1
                
                avg_val_loss = val_loss_sum / val_steps if val_steps > 0 else 0
                print(f"[{step+1}] Validation - Loss: {avg_val_loss:.4f}, MSE: {val_mse_sum/val_steps:.4f}")
                
                if wandb.run:
                    wandb.log({
                        "val_loss": avg_val_loss, 
                        "val_mse": val_mse_sum / val_steps,
                        "val_cosine": val_cos_sum / val_steps,
                        "step": step
                    })
                distiller.train()

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n[OOM] Step {step}: Cleaning cache...")
                for p in distiller.parameters():
                    if p.grad is not None: p.grad = None
                torch.cuda.empty_cache()
                import gc
                gc.collect()
                optimizer.zero_grad()
                accum_loss = 0.0; accum_mse = 0.0; accum_cosine = 0.0; accum_metrics = {}
                continue
            else: 
                print(f"Ошибка: {e}")
                continue
        except Exception as e: 
            print(f"Ошибка: {e}")
            continue

    # Сохранение после каждой эпохи
    tracker.save_checkpoint(distiller.state_dict(), name=f"phase1_{STAGE}_epoch_{epoch}")

print("Обучение завершено!")
